# Normalização e Pré-processamento do Dataset de Café
Este notebook carrega o dataset bruto, aplica limpeza e tratamento de valores ausentes, cria classes de qualidade e salva um arquivo processado para análises futuras.

### Importação de bibliotecas
Importamos `pandas` para manipulação de dados e `numpy` para operações numéricas e tratamento de valores nulos.

In [606]:
import pandas as pd
import numpy as np

### Carregamento do dataset
Lemos o arquivo CSV bruto e exibimos o número de registros e colunas para verificar rapidamente as dimensões iniciais do dataset.

In [607]:
df = pd.read_csv("../data/raw/arabica_coffee_full_table.csv")

print(f"Quantidade de registros: {df.shape[0]}")
print(f"Quantidade de colunas: {df.shape[1]}")

df.head()

Quantidade de registros: 1509
Quantidade de colunas: 42


,coffee_id,Country_of_Origin,Farm_Name,Lot_Number,Mill,ICO_Number,Company,Altitude,Region,Producer,...,Color,Category_One_Defects,Category_Two_Defects,Quakers,Expiration,Certification_Body,Certification_Address,Certification_Contact,parsed_expiration,parsed_grading_date
0,#647123,Guatemala,san francisco cotzal,11/441/50,"inmobiliaria e inversiones dos mil, s.a.",11/441/50,"inmobiliaria e inversiones dos mil, s.a.",1600.0,quiche,san francisco cotzal,...,Green,0,1,3.0,June 22 2023,Asociacion Nacional Del Café,"5a Calle 0-50, Zona 14 Guatemala City, Guatema...",Brayan Cifuentes -,2023-06-22,2022-06-22
1,#927000,Guatemala,San jose del lago,11/15/95,San jose del lago,11/15/95,"Peter Schoenfeld, S.A.",1600.0,Atitlán,"Cafetalera Paquim, S.A.",...,Green,0,2,1.0,April 16 2024,Asociacion Nacional Del Café,"5a Calle 0-50, Zona 14 Guatemala City, Guatema...",Brayan Cifuentes -,2024-04-16,2023-04-17
2,#902618,Guatemala,varias fincas,11/15/51,El Trèbol/Lìnea Gourmet,11/15/51,"Peter Schoenfeld, S.A.",1550.0,Oriente Santa rosa,varios productores,...,Green,0,2,1.0,March 21 2024,Asociacion Nacional Del Café,"5a Calle 0-50, Zona 14 Guatemala City, Guatema...",Brayan Cifuentes -,2024-03-21,2023-03-22
3,#781706,Guatemala,San jose del lago,11/15/96,San jose del lago,11/15/96,"Peter Schoenfeld, S.A.",1600.0,Atitlán,"Cafetalera Paquim, S.A.",...,Green,0,1,0.0,April 16 2024,Asociacion Nacional Del Café,"5a Calle 0-50, Zona 14 Guatemala City, Guatema...",Brayan Cifuentes -,2024-04-16,2023-04-17
4,#237025,Guatemala,Finca Alta Luz,11-63-657,NaN,11-63-657,"Retrillas del pacifico, s.a.",1350.0,Huehuetenango,Maria de los Angeles Perez,...,Green,0,5,1.0,April 25 2024,Asociacion Nacional Del Café,"5a Calle 0-50, Zona 14 Guatemala City, Guatema...",Brayan Cifuentes -,2024-04-25,2023-04-26


### Seleção de colunas relevantes
Definimos quais colunas numéricas e categóricas serão mantidas, bem como a variável alvo (`Total_Cup_Points`), para focar apenas nos dados necessários.

In [608]:
colunas_numericas = [
    "Moisture",
    "Category_One_Defects",
    "Category_Two_Defects",
    "Quakers",
    "Altitude"
]

colunas_categoricas = [
    "Variety",
    "Processing_Method",
    "Country_of_Origin",
    "Color"
]

coluna_target = "Total_Cup_Points"

df = df[colunas_numericas + colunas_categoricas + [coluna_target]].copy()

### Filtragem de registros válidos
Removemos linhas em que `Total_Cup_Points` é zero ou negativo para manter apenas avaliações válidas.

In [609]:
df = df[df["Total_Cup_Points"] > 0].copy()

print(df.shape)

(1508, 10)


In [610]:
print(f"Linhas duplicadas: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Shape após remoção: {df.shape}")

Linhas duplicadas: 1
Shape após remoção: (1507, 10)


### Tratamento de altitude inválida
Zeros em `Altitude` são tratados como valores ausentes (`NaN`), pois altitudes de 0 não são válidas para este conjunto de dados.

In [611]:
df.loc[df["Altitude"] == 0, "Altitude"] = np.nan

### Agrupamento de países menos representados
Substituímos países com menos de 10 registros pela categoria `Outros` para reduzir a cardinalidade em `Country_of_Origin`.

In [612]:
contagem = df["Country_of_Origin"].value_counts()
paises_raros = contagem[contagem < 15].index.to_list()
df["Country_of_Origin"] = df["Country_of_Origin"].replace(paises_raros, "Outros")

In [613]:
df['Color'] = df['Color'].str.strip().str.lower().str.replace('-', ' ', regex=False)
print(df['Color'].value_counts())

df['Color'] = df['Color'].replace({
    'yello green': 'yellow green',
    'browish green': 'brownish green'
})

Color
green            943
bluish green     131
blue green        94
greenish          36
yellow green      14
brownish           9
pale yellow        6
yellowish          4
yello green        1
browish green      1
yellow  green      1
Name: count, dtype: int64


In [614]:
contagem_variedades = df['Variety'].value_counts()
variedades_raras = contagem_variedades[contagem_variedades < 15].index

df['Variety'] = df['Variety'].apply(
    lambda x: 'Outros' if x in variedades_raras else x
)
print(df['Variety'].value_counts())

Variety
Caturra           279
Bourbon           241
Typica            235
Outros            112
Other             108
Catuai             83
Hawaiian Kona      44
Gesha              39
Yellow Bourbon     38
Mundo Novo         35
Catimor            31
SL14               21
SL28               17
SL34               16
Name: count, dtype: int64


### Preenchimento de valores numéricos faltantes
Para cada coluna numérica usamos a mediana como valor de preenchimento, preservando a robustez contra outliers.

In [615]:
for coluna in colunas_numericas:
    df[coluna] = df[coluna].fillna(df[coluna].median())

### Preenchimento de valores categóricos faltantes
Colunas categóricas com valores faltantes são preenchidas com `Unknown` para manter o conjunto de dados consistente.

In [616]:
for coluna in colunas_categoricas:
    df[coluna] = df[coluna].fillna("Unknown")

### Verificação final de valores ausentes
Mostramos a contagem de valores nulos após o tratamento para garantir que não existam lacunas remanescentes.

In [617]:
# 1º: padronizar texto
df['Variety'] = df['Variety'].str.strip().replace({'unknown': 'Unknown'})
df['Color'] = df['Color'].str.replace(r'\s+', ' ', regex=True).str.strip()

# 2º: SÓ DEPOIS remover duplicados
print(f"Duplicados encontrados: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Shape após remoção: {df.shape}")

Duplicados encontrados: 1
Shape após remoção: (1506, 10)


In [618]:
print(df.isnull().sum())

Moisture                0
Category_One_Defects    0
Category_Two_Defects    0
Quakers                 0
Altitude                0
Variety                 0
Processing_Method       0
Country_of_Origin       0
Color                   0
Total_Cup_Points        0
dtype: int64


### Exibição das categorias geradas
Contamos quantos exemplos há em cada classe para entender a distribuição das categorias criadas.

In [619]:
def classificar(pontos):
    if pontos < 82:
        return "Tradicional"
    elif pontos < 84:
        return "Superior"
    else:
        return "Gourmet"


df["Classe"] = df["Total_Cup_Points"].apply(classificar)

In [620]:
print(df["Classe"].value_counts())

Classe
Superior       635
Tradicional    522
Gourmet        349
Name: count, dtype: int64


### Remoção da variável original de pontuação
Depois de criar `Classe`, removemos `Total_Cup_Points` para que o dataset fique livre apenas com a variável categórica de qualidade.

In [621]:
df.drop(columns="Total_Cup_Points", inplace=True)

### Salvamento do dataset processado
Exportamos o dataset limpo e rotulado para `../data/processed/coffee_classificado.csv` para uso em análises e modelos futuros.

In [622]:
df.to_csv(
    "../data/processed/coffee_classificado.csv",
    index=False
)

print("Dataset salvo com sucesso!")

Dataset salvo com sucesso!
